# NCR Master Exploratory Data Analysis (EDA)

## 📌 Objective
This is the final, comprehensive deep-dive into both the **Sales** and **Rentals** datasets. 

**Note:** To prevent data leakage during modeling, the raw `price` column has been removed. We analyze and target `price_per_sqft` (per-unit value).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

%matplotlib inline
sns.set_theme(style="whitegrid")

# File Paths (Relative to notebooks folder)
SALES_PATH = "../data/model/model_sales.parquet"
RENTAL_PATH = "../data/model/model_rentals.parquet"

def load_data(path):
    if os.path.exists(path):
        return pd.read_parquet(path)
    return None

sales_m = load_data(SALES_PATH)
rentals_m = load_data(RENTAL_PATH)

if sales_m is not None and rentals_m is not None:
    print(f"✅ Successfully loaded {len(sales_m):,} Sales and {len(rentals_m):,} Rentals.")
else:
    print("❌ ERROR: Model datasets not found. Run data_builder.py first.")

## 1️⃣ Target Variable Distributions
Normalizing the target variable is key for XGBoost.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.histplot(np.log1p(sales_m['price_per_sqft']), kde=True, ax=ax[0], color='blue')
ax[0].set_title("Log-Scaled Sales Price/sqft")

sns.histplot(np.log1p(rentals_m['price_per_sqft']), kde=True, ax=ax[1], color='purple')
ax[1].set_title("Log-Scaled Rental Rent/sqft")
plt.show()

## 2️⃣ Micro-Market Pricing Tiers
Comparing median values across cities.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(x='city', y='price_per_sqft', data=sales_m, ax=ax[0])
ax[0].set_yscale('log')
ax[0].set_title("Sales Price by City")
ax[0].tick_params(axis='x', rotation=45)

sns.boxplot(x='city', y='price_per_sqft', data=rentals_m, ax=ax[1])
ax[1].set_yscale('log')
ax[1].set_title("Rental Rent by City")
ax[1].tick_params(axis='x', rotation=45)
plt.show()

## 3️⃣ Correlation Audit
Validating the strength of our engineered features.

In [ ]:
# Filter to only numeric columns for correlation
num_sales = sales_m.select_dtypes(include=[np.number])
num_rentals = rentals_m.select_dtypes(include=[np.number])

fig, ax = plt.subplots(1, 2, figsize=(18, 7))
sns.heatmap(num_sales.corr(), cmap='coolwarm', ax=ax[0])
ax[0].set_title("Sales Feature Correlation")

sns.heatmap(num_rentals.corr(), cmap='magma', ax=ax[1])
ax[1].set_title("Rental Feature Correlation")
plt.show()

## 4️⃣ Spatial Distribution
Physically mapping the price centroids.

In [ ]:
plt.figure(figsize=(10, 8))
plt.scatter(sales_m['longitude'], sales_m['latitude'], c=np.log1p(sales_m['price_per_sqft']), 
            alpha=0.2, s=2, cmap='viridis')
plt.colorbar(label='Log Price/sqft')
plt.title("NCR Sales Geographic Map")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.show()